# Семинар 2. Исследование методов линейной регрессии

#### Критерий оценивания:
#### Пункт 14: максимум 0,5 балла
#### Пункт 15: максимум 0,5 балла

#### Итого за работу: максимум 1 балл
#### P.S. пункты 14 и 15 без пунктов 1-13 не засчитываются, ответы на вопросы из ИИ не засчитываются

1. Загрузите датасет и выведите на экран первые несколько строк

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../first/delivery_dataset.csv')
print('Размер датасета:', df.shape)
display(df.head())

Размер датасета: (500, 8)


,Дата заказа (ГГГГ-ММ-ДД),Шифр заказа (ID),Расстояние до клиента (в км),Количество позиций в чеке (в шт.),Балл пробок на дорогах (в баллах),Погода (в градусах Цельсия),Этаж доставки (в этажах),Время доставки (в минутах)
0,2026-09-08,ORD-202608001,11.3,1,9,21.0,6,79.2
1,2026-08-29,ORD-202608002,7.3,3,7,27.0,7,46.6
2,2026-08-15,ORD-202608003,2.2,1,3,20.0,5,16.3
3,2026-08-08,ORD-202608004,0.9,2,6,22.9,1,15.8
4,2026-08-21,ORD-202608005,12.3,14,4,18.1,2,69.2


2. Разбейте выборку на признаки и ответы. Закодируйте категориальные признаки.

In [2]:
date_col = 'Дата заказа (ГГГГ-ММ-ДД)'
id_col = 'Шифр заказа (ID)'
target_col = 'Время доставки (в минутах)'

data = df.drop(columns=[id_col]).copy()
data['День недели'] = pd.to_datetime(data[date_col]).dt.day_name()
data = data.drop(columns=[date_col])

X = pd.get_dummies(
    data.drop(columns=[target_col]),
    columns=['День недели'],
    dtype=float
)
y = data[target_col].to_numpy(dtype=float)

print('Размер матрицы признаков:', X.shape)
display(X.head())

Размер матрицы признаков: (500, 12)


,Расстояние до клиента (в км),Количество позиций в чеке (в шт.),Балл пробок на дорогах (в баллах),Погода (в градусах Цельсия),Этаж доставки (в этажах),День недели_Friday,День недели_Monday,День недели_Saturday,День недели_Sunday,День недели_Thursday,День недели_Tuesday,День недели_Wednesday
0,11.3,1,9,21.0,6,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,7.3,3,7,27.0,7,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,2.2,1,3,20.0,5,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.9,2,6,22.9,1,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,12.3,14,4,18.1,2,1.0,0.0,0.0,0.0,0.0,0.0,0.0


3. Разбейте датасет на train val test в отношении 8:1:1

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print('Train:', X_train.shape, y_train.shape)
print('Validation:', X_val.shape, y_val.shape)
print('Test:', X_test.shape, y_test.shape)

Train: (400, 12) (400,)
Validation: (50, 12) (50,)
Test: (50, 12) (50,)


4. Исследуйте VGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [4]:
def loss(y_true, y_pred):
    with np.errstate(over='ignore', invalid='ignore'):
        return float(np.mean((y_true - y_pred) ** 2))

def r2(y_true, y_pred):
    return float(1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2))

def predict(X, weights):
    return np.column_stack([np.ones(len(X)), X]) @ weights

def fit_optimizer(X, y, method, learning_rate=0.01, decay=None,
                  max_iter=1000, tol=1e-9, random_state=42):
    X_with_bias = np.column_stack([np.ones(len(X)), X])
    n, p = X_with_bias.shape
    weights = np.zeros(p)
    velocity = np.zeros(p)
    first_moment = np.zeros(p)
    second_moment = np.zeros(p)
    gradient_memory = np.zeros((n, p))
    average_gradient = np.zeros(p)
    rng = np.random.default_rng(random_state)
    previous_loss = np.inf
    stable_iterations = 0
    update = 0

    for iteration in range(1, max_iter + 1):
        step = learning_rate if decay is None else learning_rate / (1 + decay * (iteration - 1))

        with np.errstate(over='ignore', invalid='ignore'):
            if method in {'VGD', 'Momentum', 'Adam'}:
                gradient = 2 / n * X_with_bias.T @ (X_with_bias @ weights - y)

                if method == 'VGD':
                    weights -= step * gradient
                elif method == 'Momentum':
                    velocity = 0.9 * velocity + gradient
                    weights -= step * velocity
                else:
                    update += 1
                    first_moment = 0.9 * first_moment + 0.1 * gradient
                    second_moment = 0.999 * second_moment + 0.001 * gradient ** 2
                    m_hat = first_moment / (1 - 0.9 ** update)
                    v_hat = second_moment / (1 - 0.999 ** update)
                    weights -= step * m_hat / (np.sqrt(v_hat) + 1e-8)
            else:
                for i in rng.permutation(n):
                    gradient_i = 2 * X_with_bias[i] * (X_with_bias[i] @ weights - y[i])
                    if method == 'SGD':
                        weights -= step * gradient_i
                    else:
                        average_gradient += (gradient_i - gradient_memory[i]) / n
                        gradient_memory[i] = gradient_i
                        weights -= step * average_gradient

            current_loss = loss(y, X_with_bias @ weights)

        if not np.isfinite(current_loss) or current_loss > 1e100:
            return weights, iteration, np.inf

        if abs(previous_loss - current_loss) < tol:
            stable_iterations += 1
        else:
            stable_iterations = 0

        if stable_iterations >= 10:
            break
        previous_loss = current_loss

    return weights, iteration, current_loss

steps = np.logspace(-5, 0, 11)
lambdas = np.logspace(-5, 0, 11)
results = []
search_history = {}

def run_experiment(method, use_decay=False):
    grid = lambdas if use_decay else steps
    trials = []

    for value in grid:
        params = {'learning_rate': 0.1, 'decay': value} if use_decay else {'learning_rate': value}
        weights, iterations, train_loss = fit_optimizer(X_train, y_train, method, **params)
        train_pred = predict(X_train, weights)
        val_loss = loss(y_val, predict(X_val, weights)) if np.isfinite(train_loss) else np.inf
        train_r2 = r2(y_train, train_pred) if np.isfinite(train_loss) else -np.inf
        trials.append({
            'lambda' if use_decay else 'n': value,
            'Loss_train': train_loss,
            'R2_train': train_r2,
            'Loss_val': val_loss,
            'weights': weights,
            'iterations': iterations
        })

    best = min(trials, key=lambda row: row['Loss_val'])
    test_pred = predict(X_test, best['weights'])
    method_name = method + (' + TimeDecay' if use_decay else '')
    step_name = (
        f'n(t) = 0.1 / (1 + {best["lambda"]:.6g}t)'
        if use_decay else f'n = {best["n"]:.6g}'
    )
    result = {
        'Метод': method_name,
        'Лучший шаг': step_name,
        'Loss_train': best['Loss_train'],
        'Loss_test': loss(y_test, test_pred),
        'R2_train': best['R2_train'],
        'R2_test': r2(y_test, test_pred),
        'Итерации': best['iterations']
    }
    results.append(result)
    history = pd.DataFrame(trials).drop(columns=['weights'])
    search_history[method_name] = history
    display(history.round(6))
    print('Лучший вариант:')
    display(pd.DataFrame([result]).round(6))

run_experiment('VGD')

/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:31: RuntimeWarning: divide by zero encountered in matmul
  gradient = 2 / n * X_with_bias.T @ (X_with_bias @ weights - y)
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:55: RuntimeWarning: divide by zero encountered in matmul
  current_loss = loss(y, X_with_bias @ weights)
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: divide by zero encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: overflow encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: invalid value encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights


,n,Loss_train,R2_train,Loss_val,iterations
0,0.000010,2467.034525,-2.435026,1944.118809,1000
1,0.000032,2262.846843,-2.150721,1784.122990,1000
2,0.000100,1723.493802,-1.399742,1360.343901,1000
3,0.000316,736.998575,-0.026175,579.950001,1000
4,0.001000,68.328696,0.904861,48.336645,1000
5,0.003162,19.965536,0.972201,15.170166,1000
6,0.010000,19.951635,0.972220,15.330767,701
7,0.031623,19.951635,0.972220,15.330872,235
8,0.100000,19.951635,0.972220,15.330927,80
9,0.316228,19.951635,0.972220,15.330936,28


Лучший вариант:


,Метод,Лучший шаг,Loss_train,Loss_test,R2_train,R2_test,Итерации
0,VGD,n = 0.00316228,19.965536,20.100947,0.972201,0.967125,1000


5. Исследуйте VGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [5]:
run_experiment('VGD', use_decay=True)

/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:31: RuntimeWarning: divide by zero encountered in matmul
  gradient = 2 / n * X_with_bias.T @ (X_with_bias @ weights - y)
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:55: RuntimeWarning: divide by zero encountered in matmul
  current_loss = loss(y, X_with_bias @ weights)
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: divide by zero encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: overflow encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: invalid value encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights


,lambda,Loss_train,R2_train,Loss_val,iterations
0,0.000010,19.951635,0.972220,15.330927,80
1,0.000032,19.951635,0.972220,15.330926,80
2,0.000100,19.951635,0.972220,15.330928,81
3,0.000316,19.951635,0.972220,15.330926,81
4,0.001000,19.951635,0.972220,15.330926,83
5,0.003162,19.951635,0.972220,15.330922,89
6,0.010000,19.951635,0.972220,15.330901,111
7,0.031623,19.951635,0.972220,15.330787,243
8,0.100000,19.951686,0.972220,15.321300,1000
9,0.316228,21.418059,0.970178,14.925843,1000


Лучший вариант:


,Метод,Лучший шаг,Loss_train,Loss_test,R2_train,R2_test,Итерации
0,VGD + TimeDecay,n(t) = 0.1 / (1 + 0.316228t),21.418059,17.349956,0.970178,0.971624,1000


6. Исследуйте SGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [6]:
run_experiment('SGD')

/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:55: RuntimeWarning: divide by zero encountered in matmul
  current_loss = loss(y, X_with_bias @ weights)


/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: divide by zero encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: overflow encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: invalid value encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights


,n,Loss_train,R2_train,Loss_val,iterations
0,0.000010,19.952360,0.972219,15.293274,1000
1,0.000032,19.951635,0.972220,15.330522,1000
2,0.000100,19.951658,0.972220,15.324102,1000
3,0.000316,19.952429,0.972219,15.280377,1000
4,0.001000,19.966678,0.972199,15.116200,1000
5,0.003162,20.246261,0.971810,15.172437,1000
6,0.010000,22.801856,0.968251,17.635740,1000
7,0.031623,54.305815,0.924386,52.064316,1000
8,0.100000,inf,-inf,inf,5
9,0.316228,inf,-inf,inf,1


Лучший вариант:


,Метод,Лучший шаг,Loss_train,Loss_test,R2_train,R2_test,Итерации
0,SGD,n = 0.001,19.966678,20.576952,0.972199,0.966347,1000


7. Исследуйте SGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [7]:
run_experiment('SGD', use_decay=True)

/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:55: RuntimeWarning: divide by zero encountered in matmul
  current_loss = loss(y, X_with_bias @ weights)
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: divide by zero encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: overflow encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: invalid value encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights


,lambda,Loss_train,R2_train,Loss_val,iterations
0,0.000010,inf,-inf,inf,5
1,0.000032,inf,-inf,inf,5
2,0.000100,inf,-inf,inf,5
3,0.000316,inf,-inf,inf,5
4,0.001000,inf,-inf,inf,5
5,0.003162,inf,-inf,inf,5
6,0.010000,inf,-inf,inf,5
7,0.031623,inf,-inf,inf,6
8,0.100000,19.966368,0.972199,15.117761,1000
9,0.316228,19.952429,0.972219,15.280436,1000


Лучший вариант:


,Метод,Лучший шаг,Loss_train,Loss_test,R2_train,R2_test,Итерации
0,SGD + TimeDecay,n(t) = 0.1 / (1 + 0.1t),19.966368,20.574072,0.972199,0.966351,1000


8. Исследуйте SAG с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [8]:
run_experiment('SAG')

/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:55: RuntimeWarning: divide by zero encountered in matmul
  current_loss = loss(y, X_with_bias @ weights)


/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: divide by zero encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: overflow encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: invalid value encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights


,n,Loss_train,R2_train,Loss_val,iterations
0,0.000010,19.952322,0.972219,15.294157,1000
1,0.000032,19.951635,0.972220,15.330793,560
2,0.000100,19.951635,0.972220,15.330889,188
3,0.000316,19.951635,0.972220,15.330932,64
4,0.001000,19.951635,0.972220,15.330936,25
5,0.003162,19.951635,0.972220,15.330935,43
6,0.010000,inf,-inf,inf,313
7,0.031623,inf,-inf,inf,315
8,0.100000,inf,-inf,inf,134
9,0.316228,inf,-inf,inf,53


Лучший вариант:


,Метод,Лучший шаг,Loss_train,Loss_test,R2_train,R2_test,Итерации
0,SAG,n = 1e-05,19.952322,20.44184,0.972219,0.966568,1000


9. Исследуйте SAG с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [9]:
run_experiment('SAG', use_decay=True)

/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:55: RuntimeWarning: divide by zero encountered in matmul
  current_loss = loss(y, X_with_bias @ weights)
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: divide by zero encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: overflow encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: invalid value encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights


,lambda,Loss_train,R2_train,Loss_val,iterations
0,0.000010,inf,-inf,inf,134
1,0.000032,inf,-inf,inf,135
2,0.000100,inf,-inf,inf,138
3,0.000316,inf,-inf,inf,139
4,0.001000,inf,-inf,inf,146
5,0.003162,inf,-inf,inf,165
6,0.010000,inf,-inf,inf,235
7,0.031623,inf,-inf,inf,325
8,0.100000,2.048309e+20,-2.852004e+17,1.695136e+20,1000
9,0.316228,1.995163e+01,9.722200e-01,1.533094e+01,153


Лучший вариант:


,Метод,Лучший шаг,Loss_train,Loss_test,R2_train,R2_test,Итерации
0,SAG + TimeDecay,n(t) = 0.1 / (1 + 0.316228t),19.951635,20.538536,0.97222,0.966409,153


10. Исследуйте Momentum с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [10]:
run_experiment('Momentum')

/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:31: RuntimeWarning: divide by zero encountered in matmul
  gradient = 2 / n * X_with_bias.T @ (X_with_bias @ weights - y)
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:55: RuntimeWarning: divide by zero encountered in matmul
  current_loss = loss(y, X_with_bias @ weights)
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: divide by zero encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: overflow encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: invalid value encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights


,n,Loss_train,R2_train,Loss_val,iterations
0,0.000010,1728.436150,-1.406623,1364.261445,1000
1,0.000032,740.028807,-0.030394,582.465251,1000
2,0.000100,66.675689,0.907163,47.125466,1000
3,0.000316,19.959683,0.972209,15.208057,1000
4,0.001000,19.951635,0.972220,15.330805,588
5,0.003162,19.951635,0.972220,15.331023,245
6,0.010000,19.951635,0.972220,15.330988,255
7,0.031623,19.951635,0.972220,15.331023,263
8,0.100000,19.951635,0.972220,15.330974,265
9,0.316228,19.951635,0.972220,15.330919,277


Лучший вариант:


,Метод,Лучший шаг,Loss_train,Loss_test,R2_train,R2_test,Итерации
0,Momentum,n = 0.000316228,19.959683,20.207178,0.972209,0.966951,1000


11. Исследуйте Momentum с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [11]:
run_experiment('Momentum', use_decay=True)

/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:31: RuntimeWarning: divide by zero encountered in matmul
  gradient = 2 / n * X_with_bias.T @ (X_with_bias @ weights - y)
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:55: RuntimeWarning: divide by zero encountered in matmul
  current_loss = loss(y, X_with_bias @ weights)
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: divide by zero encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: overflow encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: invalid value encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights


,lambda,Loss_train,R2_train,Loss_val,iterations
0,0.000010,19.951635,0.97222,15.330973,265
1,0.000032,19.951635,0.97222,15.330974,266
2,0.000100,19.951635,0.97222,15.330971,267
3,0.000316,19.951635,0.97222,15.330893,263
4,0.001000,19.951635,0.97222,15.330963,266
5,0.003162,19.951635,0.97222,15.330922,262
6,0.010000,19.951635,0.97222,15.330884,260
7,0.031623,19.951635,0.97222,15.330855,249
8,0.100000,19.951635,0.97222,15.330994,236
9,0.316228,19.951635,0.97222,15.330946,251


Лучший вариант:


,Метод,Лучший шаг,Loss_train,Loss_test,R2_train,R2_test,Итерации
0,Momentum + TimeDecay,n(t) = 0.1 / (1 + 0.0316228t),19.951635,20.538393,0.97222,0.96641,249


12. Исследуйте Adam с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [12]:
run_experiment('Adam')

/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:31: RuntimeWarning: divide by zero encountered in matmul
  gradient = 2 / n * X_with_bias.T @ (X_with_bias @ weights - y)
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:55: RuntimeWarning: divide by zero encountered in matmul
  current_loss = loss(y, X_with_bias @ weights)
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: divide by zero encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: overflow encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: invalid value encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights


,n,Loss_train,R2_train,Loss_val,iterations
0,0.000010,2565.921883,-2.572713,2022.122510,1000
1,0.000032,2562.057013,-2.567332,2020.388258,1000
2,0.000100,2549.958270,-2.550486,2014.977833,1000
3,0.000316,2512.813913,-2.498767,1998.427254,1000
4,0.001000,2404.453473,-2.347890,1948.201869,1000
5,0.003162,2116.045313,-1.946319,1767.594973,1000
6,0.010000,1415.417234,-0.970785,1200.006906,1000
7,0.031623,333.319809,0.535895,282.377899,1000
8,0.100000,20.158861,0.971931,15.018670,1000
9,0.316228,19.951635,0.972220,15.330861,564


Лучший вариант:


,Метод,Лучший шаг,Loss_train,Loss_test,R2_train,R2_test,Итерации
0,Adam,n = 0.1,20.158861,19.858961,0.971931,0.967521,1000


13. Исследуйте Adam с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [13]:
run_experiment('Adam', use_decay=True)

/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:31: RuntimeWarning: divide by zero encountered in matmul
  gradient = 2 / n * X_with_bias.T @ (X_with_bias @ weights - y)
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:55: RuntimeWarning: divide by zero encountered in matmul
  current_loss = loss(y, X_with_bias @ weights)
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: divide by zero encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: overflow encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights
/var/folders/5h/4zk3c1qs3dq8s5sr2pycz_r00000gn/T/ipykernel_26148/1530381014.py:9: RuntimeWarning: invalid value encountered in matmul
  return np.column_stack([np.ones(len(X)), X]) @ weights


,lambda,Loss_train,R2_train,Loss_val,iterations
0,0.000010,20.172253,0.971913,15.015496,1000
1,0.000032,20.203425,0.971869,15.009945,1000
2,0.000100,20.323865,0.971702,15.006278,1000
3,0.000316,20.984349,0.970782,15.199130,1000
4,0.001000,27.583284,0.961594,19.712422,1000
5,0.003162,106.770167,0.851336,87.904712,1000
6,0.010000,528.142702,0.264630,449.973482,1000
7,0.031623,1299.236757,-0.809019,1102.513624,1000
8,0.100000,1931.077807,-1.688775,1629.609837,1000
9,0.316228,2277.180752,-2.170679,1885.438043,1000


Лучший вариант:


,Метод,Лучший шаг,Loss_train,Loss_test,R2_train,R2_test,Итерации
0,Adam + TimeDecay,n(t) = 0.1 / (1 + 0.0001t),20.323865,19.71983,0.971702,0.967748,1000


14. Постройте итоговую сравнительную таблицу со следующими столбцами:

1) название метода

2) значение лучшего шага (если n) или функция лучшего шага (если n(lyamda))

3) Loss_train

4) Loss_test

5) R^2 train

6) R^2 test

7) число итераций на test

In [14]:
comparison = pd.DataFrame(results)
comparison = comparison.sort_values('Loss_test').reset_index(drop=True)
display(comparison.round({
    'Loss_train': 4,
    'Loss_test': 4,
    'R2_train': 4,
    'R2_test': 4
}))

,Метод,Лучший шаг,Loss_train,Loss_test,R2_train,R2_test,Итерации
0,VGD + TimeDecay,n(t) = 0.1 / (1 + 0.316228t),21.4181,17.3500,0.9702,0.9716,1000
1,Adam + TimeDecay,n(t) = 0.1 / (1 + 0.0001t),20.3239,19.7198,0.9717,0.9677,1000
2,Adam,n = 0.1,20.1589,19.8590,0.9719,0.9675,1000
3,VGD,n = 0.00316228,19.9655,20.1009,0.9722,0.9671,1000
4,Momentum,n = 0.000316228,19.9597,20.2072,0.9722,0.9670,1000
5,SAG,n = 1e-05,19.9523,20.4418,0.9722,0.9666,1000
6,Momentum + TimeDecay,n(t) = 0.1 / (1 + 0.0316228t),19.9516,20.5384,0.9722,0.9664,249
7,SAG + TimeDecay,n(t) = 0.1 / (1 + 0.316228t),19.9516,20.5385,0.9722,0.9664,153
8,SGD + TimeDecay,n(t) = 0.1 / (1 + 0.1t),19.9664,20.5741,0.9722,0.9664,1000
9,SGD,n = 0.001,19.9667,20.5770,0.9722,0.9663,1000


15. Сделайте вывод о том, какой метод и шаг линейной регрессии самый лучший для данной выборки и ответьте на вопросы:

1) почему именно этот метод и этот шаг самый лучший (по каким данным из таблицы вы сделали такой вывод)

2) расскажите простыми словами суть R^2_train и R^2_test?

3) как R^2_train и R^2_test помогают сравнивать методы? почему оба эти значения надо вычислять для данного выбора?

лучше всего получился VGD с переменным шагом n(t) = 0.1 / (1 + 0.316228t). у него самый маленький Loss_test 17.35 и самый большой R^2_test 0.972, то есть на новых данных он ошибается меньше остальных.  
R^2_train показывает, какую долю разброса ответов модель объяснила на данных, на которых училась. R^2_test показывает то же самое на новых данных. чем R^2 ближе к 1, тем лучше.  
по R^2_train видно, насколько хорошо метод выучил train, а по R^2_test — переносится ли это качество на новые данные. нужны оба значения: высокий R^2_train при заметно меньшем R^2_test может означать переобучение.